## NA2M - full pipeline: arm A vs arm B (GAMI-Net) vs arm C (NA2M)

In [1]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

SEED = 42
set_seed(SEED)

### Config

In [2]:
from na2m.utils.config import load_na2m_config

config = load_na2m_config(r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas-scores-two-years_tuned.yaml')
config.top_m = 20
config.clarity_regularization = 0.1
config.eta_prune = 0.01

print(config)

NA2MConfig(dataset_path='C:\\Users\\maart\\OneDrive\\Documenten\\Universiteit\\Scriptie\\python_repo\\thesis-nam\\datasets\\raw\\compas-scores-two-years.csv', num_units=64, hidden_sizes=[64, 32], activation='relu', dropout=0.1, inter_units=32, inter_hidden=[], feature_dropout=0.05, output_regularization=0.03242773404285635, l2_regularization=3.002585238683321e-05, clarity_regularization=0.1, lr=0.027365104862115727, decay_rate=0.995, val_frac=0.15, test_frac=0.15, seed=42, pool_val_frac=0.15, batch_size=1024, num_epochs=200, patience=50, val_check_interval=10, top_m=20, eta_prune=0.01, block_train_epochs=1000, finetune_epochs=100, concurvity_filter=True, concurvity_threshold=0.5, k_folds=5, fold_seed=42, seeds=[0, 1, 2, 3, 4], grid_size=100, task='classification')


### Data

In [3]:
from na2m.data.data_utils import load_compas, preprocess, split

df = load_compas(r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\datasets\raw\compas-scores-two-years.csv')
X, y, feature_meta = preprocess(df)

print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}')
for fm in feature_meta:
    print(f'  {fm.name:25s} type={fm.type}')

Samples: 6907, Features: 6
  age                       type=num
  priors_count              type=num
  length_of_stay            type=num
  c_charge_degree           type=cat
  race                      type=cat
  sex                       type=cat


### Loaders

In [4]:
from nam.data.dataset import NAMDataset
from torch.utils.data import DataLoader

X_train, X_val, X_test, y_train, y_val, y_test = split(X, y, val_frac=0.15, test_frac=0.15, seed=SEED)

train_dataset = NAMDataset(X_train, y_train)
val_dataset   = NAMDataset(X_val, y_val)
pool_dataset  = NAMDataset(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
test_dataset  = NAMDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=config.batch_size, shuffle=False)
pool_loader  = DataLoader(pool_dataset,  batch_size=config.batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=config.batch_size, shuffle=False)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Pool: {len(pool_dataset)}, Test: {len(test_dataset)}')

Train: 4834, Val: 1036, Pool: 5870, Test: 1037


### Helpers

In [5]:
from na2m.models.na2m import NA2M
from na2m.training.fit_na2m import fit_na2m
from nam.training.metrics import auroc

def build_model():
    return NA2M(
        num_features=X.shape[1], feature_meta=feature_meta,
        num_units=config.num_units, hidden_sizes=config.hidden_sizes,
        dropout=config.dropout, feature_dropout=config.feature_dropout,
        activation=config.activation, inter_units=config.inter_units,
        inter_hidden=config.inter_hidden,
    )

def test_auroc(model):
    model.eval()
    logits_all, targets_all = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:
            logits, _ = model(X_batch)
            logits_all.append(logits)
            targets_all.append(y_batch)
    return auroc(torch.cat(logits_all), torch.cat(targets_all))

def print_result(label, auc, result):
    pairs = result['active_pairs']
    print(f'{label}  AUROC={auc:.4f}  interactions={len(pairs)}')
    for j, k in pairs:
        print(f'  {feature_meta[j].name} x {feature_meta[k].name}')

### Arm A - mains only

In [6]:
set_seed(SEED)
model_a = build_model()

result_a = fit_na2m(
    model_a, train_loader, val_loader, pool_loader, config,
    with_interactions=False,
    with_concurvity_filter=False,
)

auc_a = test_auroc(model_a)
print_result('Arm A (mains only)', auc_a, result_a)

Epoch 1/200| Epoch loss = 0.7256 | best=0.0000
Epoch 100/200| Epoch loss = 0.6192 | best=0.7332
Early stopping at epoch 160
Arm A (mains only)  AUROC=0.7564  interactions=0


### Arm B - GAMI-Net (interactions, no concurvity filter)

In [7]:
set_seed(SEED)
model_b = build_model()

result_b = fit_na2m(
    model_b, train_loader, val_loader, pool_loader, config,
    with_interactions=True,
    with_concurvity_filter=False,
)

auc_b = test_auroc(model_b)
print_result('Arm B (GAMI-Net, no filter)', auc_b, result_b)

Epoch 1/200| Epoch loss = 0.7256 | best=0.0000
Epoch 100/200| Epoch loss = 0.6192 | best=0.7332
Early stopping at epoch 160
[stage2] FAST screen: top-15 pairs selected from 15 candidates
[stage2] Block-training 15 interaction subnets for 1000 epochs...
Epoch 1/1000| Epoch loss = 0.7186 | best=0.0000
Early stopping at epoch 100
[stage2] Block-train done. Best val metric: 0.7360
[stage2] Contribution ranking (variance, descending):
  (4,5)  var=0.006248
  (3,4)  var=0.005887
  (0,4)  var=0.002632
  (0,2)  var=0.001300
  (0,5)  var=0.000479
  (0,1)  var=0.000383
  (1,4)  var=0.000263
  (2,4)  var=0.000154
  (1,5)  var=0.000060
  (1,2)  var=0.000049
  (2,5)  var=0.000012
  (0,3)  var=0.000008
  (2,3)  var=0.000007
  (3,5)  var=0.000004
  (1,3)  var=0.000001
[stage2] Sweep baseline val loss (mains only): 0.60287
  pair (4, 5) — accepted  val_loss=0.60242
  pair (3, 4) — accepted  val_loss=0.60232
  pair (0, 4) — accepted  val_loss=0.60250
  pair (0, 2) — accepted  val_loss=0.60135
  pair (0

### Arm C - NA2M (interactions + concurvity filter)

In [8]:
set_seed(SEED)
model_c = build_model()

result_c = fit_na2m(
    model_c, train_loader, val_loader, pool_loader, config,
    with_interactions=True,
    with_concurvity_filter=True,
)

auc_c = test_auroc(model_c)
print_result('Arm C (NA2M, with filter)', auc_c, result_c)

Epoch 1/200| Epoch loss = 0.7256 | best=0.0000
Epoch 100/200| Epoch loss = 0.6192 | best=0.7332
Early stopping at epoch 160
[stage2] FAST screen: top-15 pairs selected from 15 candidates
[stage2] Block-training 15 interaction subnets for 1000 epochs...
Epoch 1/1000| Epoch loss = 0.7186 | best=0.0000
Early stopping at epoch 100
[stage2] Block-train done. Best val metric: 0.7360
[stage2] Contribution ranking (variance, descending):
  (4,5)  var=0.006248
  (3,4)  var=0.005887
  (0,4)  var=0.002632
  (0,2)  var=0.001300
  (0,5)  var=0.000479
  (0,1)  var=0.000383
  (1,4)  var=0.000263
  (2,4)  var=0.000154
  (1,5)  var=0.000060
  (1,2)  var=0.000049
  (2,5)  var=0.000012
  (0,3)  var=0.000008
  (2,3)  var=0.000007
  (3,5)  var=0.000004
  (1,3)  var=0.000001
[stage2] Sweep baseline val loss (mains only): 0.60287
  pair (4, 5) — accepted  val_loss=0.60242
  pair (3, 4) — accepted  val_loss=0.60232
  pair (0, 4) — accepted  val_loss=0.60250
  pair (0, 2) — accepted  val_loss=0.60135
  pair (0

### Comparison

In [9]:
print(f'Arm A (mains only):          AUROC={auc_a:.4f}  interactions=0')
print(f'Arm B (GAMI-Net, no filter): AUROC={auc_b:.4f}  interactions={len(result_b["active_pairs"])}')
print(f'Arm C (NA2M, with filter):   AUROC={auc_c:.4f}  interactions={len(result_c["active_pairs"])}')

Arm A (mains only):          AUROC=0.7564  interactions=0
Arm B (GAMI-Net, no filter): AUROC=0.7507  interactions=14
Arm C (NA2M, with filter):   AUROC=0.7518  interactions=11
